In [1]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,                        # loads the right tokenizer for any model
    AutoModelForSequenceClassification,   # loads DistilBERT + adds classification head
    TrainingArguments,                    # config object: epochs, batch size, LR, etc.
    Trainer,                              # HuggingFace training loop (handles backprop, etc.)
)
from sklearn.metrics import f1_score
import evaluate  # HuggingFace's metrics library (separate from transformers)
import torch
from transformers import DataCollatorWithPadding
from transformers import pipeline

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training will run on: {device.upper()}")
print(f"PyTorch version: {torch.__version__}")

Training will run on: CUDA
PyTorch version: 2.12.0+cu130


In [3]:
dataset = load_dataset("mteb/banking77")

# sorted() ensures our mapping is deterministic and alphabetical.
label_names = sorted(set(dataset["train"]["label_text"]))  # list of 77 unique strings
#dictionaries totokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT) translate back and forth between integers and human-readable class names
label2id = {name: idx for idx, name in enumerate(label_names)}
id2label  = {idx: name for name, idx in label2id.items()}

print(f"Number of unique labels: {len(label2id)}")
print(f"\nFirst 5 label mappings:")
for name, idx in list(label2id.items())[:5]:
    print(f"  '{name}' → {idx}")

Number of unique labels: 77

First 5 label mappings:
  'Refund_not_showing_up' → 0
  'activate_my_card' → 1
  'age_limit' → 2
  'apple_pay_or_google_pay' → 3
  'atm_support' → 4


In [4]:
#Load Tokenizer

MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

In [5]:
def tokenize_and_encode(batch):
    """
    Called once per batch of examples by dataset.map().
    
    'batch' is a dict where each key maps to a LIST of values:
      batch["text"]       = ["I lost my card", "What's my balance?", ...]
      batch["label_text"] = ["card_not_working", "balance_enquiry", ...]
    
    We return a dict of the tokenizer outputs + our labels.
    The labels key must be named "labels" (exactly) for HuggingFace
    Trainer to pick it up automatically.
    """
    # tokenizer() called with a list of strings processes them all at once
    tokenized = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    # Re-encode labels using OUR label2id mapping (not the dataset's raw integers).
    # We go through label_text (the string) rather than label (the int) so we
    # control the mapping ourselves.
    tokenized["labels"] = [label2id[text] for text in batch["label_text"]]
    return tokenized

print("Tokenizing dataset... (this takes 30-60 seconds)")

tokenized_dataset = dataset.map(
    tokenize_and_encode,
    batched=True,
    # Remove columns the Trainer doesn't need. If we leave them in,
    # Trainer will try to pass them to the model and crash.
    remove_columns=["text", "label", "label_text"],
)

# Tell the dataset to return PyTorch tensors when indexed.
# Without this, it returns Python lists, and PyTorch can't batch them.
tokenized_dataset.set_format("torch")

print("\nDataset structure:")
print(tokenized_dataset)
print(f"\nOne training example:")
sample = tokenized_dataset["train"][0]
for key, val in sample.items():
    print(f"  {key}: shape={val.shape}, dtype={val.dtype}")

Tokenizing dataset... (this takes 30-60 seconds)

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 9993
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3076
    })
})

One training example:
  input_ids: shape=torch.Size([128]), dtype=torch.int64
  token_type_ids: shape=torch.Size([128]), dtype=torch.int64
  attention_mask: shape=torch.Size([128]), dtype=torch.int64
  labels: shape=torch.Size([]), dtype=torch.int64


In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label2id),   # tells the model how many classes to output
    id2label=id2label,          # bakes label names into the model config
    label2id=label2id,          # bakes label2id into the model config too
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
total_params     = model.num_parameters()
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nDistilBERT is ~40% smaller than BERT (66M vs 110M params)")
print(f"while keeping ~97% of BERT's performance — good tradeoff for a sprint.")

Total parameters:     67,012,685
Trainable parameters: 67,012,685

DistilBERT is ~40% smaller than BERT (66M vs 110M params)
while keeping ~97% of BERT's performance — good tradeoff for a sprint.


In [8]:
accuracy_metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    """
    Called by Trainer after each evaluation pass.
    
    eval_pred is a tuple: (logits, labels)
      logits: raw model outputs, shape (num_examples, 77)
              NOT probabilities yet — we need argmax to get predictions
      labels: true integer class IDs, shape (num_examples,)
    """
    logits, labels = eval_pred

    # argmax picks the index of the highest logit value = predicted class
    # axis=-1 means "take argmax along the last dimension" (across 77 classes)
    predictions = np.argmax(logits, axis=-1)

    # HuggingFace evaluate library handles accuracy
    acc_result = accuracy_metric.compute(predictions=predictions, references=labels)

    # sklearn handles macro F1 — average="macro" = equal weight per class
    f1 = f1_score(labels, predictions, average="macro")

    return {
        "accuracy": acc_result["accuracy"],
        "f1_macro": f1,
    }

In [9]:
training_args = TrainingArguments(
    output_dir="../models/banking77-distilbert/checkpoints",
    num_train_epochs=8,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    warmup_steps=0.05,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_dir="../models/banking77-distilbert/logs",
    logging_steps=50,
    report_to="none",
    fp16=True,
)

print("TrainingArguments configured. Key settings:")
print(f"  Epochs:           {training_args.num_train_epochs}")
print(f"  Train batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate:    {training_args.learning_rate}")
print(f"  Best model by:    {training_args.metric_for_best_model}")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


TrainingArguments configured. Key settings:
  Epochs:           8
  Train batch size: 32
  Learning rate:    3e-05
  Best model by:    f1_macro


In [10]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

#Training
print("Starting training...")
print("Expected output: one eval block per epoch showing eval_loss, accuracy, f1_macro")
print("─" * 60)

# On CPU:  ~45-90 minutes for 3 epochs
# On GPU:  ~5-10 minutes for 3 epochs
train_result = trainer.train()

print("─" * 60)
print("Training complete!")
print(f"Training took: {train_result.metrics['train_runtime']:.0f} seconds")
print(f"Samples/second: {train_result.metrics['train_samples_per_second']:.1f}")

Starting training...
Expected output: one eval block per epoch showing eval_loss, accuracy, f1_macro
────────────────────────────────────────────────────────────


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,2.439470,2.087613,0.687906,0.645976
2,0.937543,0.778103,0.855007,0.845631
3,0.427993,0.437700,0.910598,0.910359
4,0.219214,0.344868,0.913849,0.914000
5,0.158364,0.294307,0.925878,0.925897
6,0.114224,0.291252,0.926203,0.926122
7,0.073737,0.280876,0.929454,0.929468
8,0.063983,0.278020,0.930429,0.930362


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

────────────────────────────────────────────────────────────
Training complete!
Training took: 141 seconds
Samples/second: 565.5


In [11]:
print("Running final evaluation on test set...")
results = trainer.evaluate()

print("\n" + "═" * 40)
print("FINAL EVALUATION RESULTS")
print("═" * 40)
for metric, value in results.items():
    # Format floats to 4 decimal places, keep everything else as-is
    if isinstance(value, float):
        print(f"  {metric:<30} {value:.4f}")
    else:
        print(f"  {metric:<30} {value}")

Running final evaluation on test set...


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.063983,0.278020,8,0.930429,0.930362



════════════════════════════════════════
FINAL EVALUATION RESULTS
════════════════════════════════════════
  eval_loss                      0.2780
  eval_accuracy                  0.9304
  eval_f1_macro                  0.9304


In [12]:
SAVE_PATH = "../models/banking77-distilbert"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Model and tokenizer saved to: {SAVE_PATH}/")
print("\nSaved files:")

import os
for f in sorted(os.listdir(SAVE_PATH)):
    size_kb = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1024
    print(f"  {f:<35} ({size_kb:.1f} KB)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to: /home/kaze/nlp/document-classifier/model/banking77-distilbert/

Saved files:
  config.json                         (5.7 KB)
  model.safetensors                   (261780.5 KB)
  tokenizer.json                      (694.7 KB)
  tokenizer_config.json               (0.3 KB)
  training_args.bin                   (5.2 KB)


In [13]:
print("Loading saved model for smoke test...")

classifier = pipeline(
    "text-classification",
    model=SAVE_PATH,
    tokenizer=SAVE_PATH,
    device=0 if torch.cuda.is_available() else -1,  # 0 = first GPU, -1 = CPU
)

test_queries = [
    "I can't find my card anywhere",
    "What's my current balance?",
    "My payment was declined at the store",
    "I want to transfer money to another account",
    "Why was I charged twice for this transaction?",
    "How do I change my PIN?",
]

print("\n" + "═" * 60)
print("SMOKE TEST — Manual Predictions")
print("═" * 60)
for query in test_queries:
    result = classifier(query)[0]
    label  = result["label"]
    score  = result["score"]
    print(f"\n  Query:      '{query}'")
    print(f"  Predicted:  {label}")
    print(f"  Confidence: {score:.1%}")

print("\n" + "═" * 60)

Loading saved model for smoke test...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════
SMOKE TEST — Manual Predictions
════════════════════════════════════════════════════════════

  Query:      'I can't find my card anywhere'
  Predicted:  lost_or_stolen_card
  Confidence: 78.8%

  Query:      'What's my current balance?'
  Predicted:  balance_not_updated_after_bank_transfer
  Confidence: 97.7%

  Query:      'My payment was declined at the store'
  Predicted:  declined_card_payment
  Confidence: 97.8%

  Query:      'I want to transfer money to another account'
  Predicted:  transfer_into_account
  Confidence: 98.0%

  Query:      'Why was I charged twice for this transaction?'
  Predicted:  transaction_charged_twice
  Confidence: 99.4%

  Query:      'How do I change my PIN?'
  Predicted:  change_pin
  Confidence: 99.3%

════════════════════════════════════════════════════════════
